In [1]:
"""
exp07 Step 02: Continuous Replay On The Validation Span (PROTOCOL.md §6-§9)

One continuous path over the 44 validation quarters of the exp02-exp06 schedule (2015-04-17 -> 2026-04-15); the position
carries across quarters and nothing forces a buy-back.
    1. CANDIDATE PATHS (after the fact, optimistic for the best): the 3 candidates, each replayed over the whole span; its
       return in each period minus buy-and-hold's = its period score (3 x 44 = 132 validation trials).
    2. SELECTION: mean period score over the last 4 periods; ties: the larger k (fewer exits).
    3. PRIOR-ONLY PATH (the honest estimate): the candidate selected at period f governs period f + 1; one continuous
       switching path from period 1 to the end, with buy-and-hold, 20 random-exit and 20 random-re-entry runs on the
       same sessions, the success criteria, the log-excess interval, the cost sensitivity and the episode scorecard.

STOP (PROTOCOL.md §9) if the prior-only path does not beat buy-and-hold, or does not beat the median of both random
families. No test window is evaluated: the bars are cut at the last validation session before any simulation.
RUN_MODE: "validation_only" only (the untouched window 2026-05-14 -> 2026-08-13 is never read for evaluation).
Prerequisite: the raw data only. Outputs: store04_experiments/exp07_warning_lights_exit/step02_continuous_replay_data/.
"""
None

In [2]:
# IMPORT GENERAL PACKAGES
import os
import pandas as pd
import numpy as np
import time
# IMPORT PLOTLY
import plotly.graph_objects as go
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp07_warning_lights_exit import config as exp_config
# IMPORT DATA LOADING, EXECUTION AND DAILY FEATURE FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
from so.features.daily_features import get_daily_feature_pdf
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT THE CONTINUOUS REPLAY AND WALK-FORWARD FUNCTIONS
from so.core.continuous_replay import get_replay_period_pdf
from so.core.replay_walk_forward import run_candidate_replay_dict, run_prior_only_replay_dict
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp07_warning_lights_exit.rules import get_schedule_tuple, get_rule_candidate_list, get_rule_builder_func
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_pdf, log_trial_dict, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [3]:
# DEFINE THE RUN MODE: ONLY "validation_only" EXISTS FOR THIS EXPERIMENT (NO TEST WINDOW IS EVALUATED)
RUN_MODE_STR = "validation_only"
assert RUN_MODE_STR == "validation_only"
# DEFINE THE CANDIDATE KEYS AND THE TIE-BREAKS OF THE SELECTION (PROTOCOL.md §6: LARGER k)
KEY_COL_STR_LIST = ["light_k"]
TIE_BREAK_LIST = [("light_k", False)]
# DISPLAY THE SETTINGS
print(f"Run Mode:\t{RUN_MODE_STR}\tLights:\t{list(exp_config.LIGHT_DICT)}\tk:\t{exp_config.LIGHT_K_LIST}\tMax cash sessions:\t{exp_config.MAX_CASH_SESSIONS}")

Run Mode:	validation_only	Lights:	['below_ma200', 'return_250d_negative', 'volatility_rising', 'ath_drawdown_10pct', 'high60_stale']	k:	[3, 4, 5]	Max cash sessions:	None


In [4]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step02_continuous_replay_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "validation")

Experiment:	exp07_warning_lights_exit	Protocol:	1.0	Config Hash:	2965fad3e3
Output folder:	C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp07_warning_lights_exit/step02_continuous_replay_data/
ℹ️ No earlier 'validation' entry of exp07_warning_lights_exit in the trial log (config hash 2965fad3e3).


In [5]:
# CALL FUNCTION TO READ THE MINUTE BARS AND BUILD THE SCHEDULE FROM THE FULL SESSION CALENDAR (SAME 44 QUARTERS AS exp02-exp06)
complete_ohlcv_pdf = get_complete_ohlcv_pdf()
fold_pdf, _ = get_schedule_tuple(sorted(complete_ohlcv_pdf["date"].unique()))
# DEFINE THE LAST VALIDATION SESSION AND CUT THE BARS THERE (NO LATER BAR IS GIVEN TO THE SIMULATOR)
replay_end_date = pd.to_datetime(fold_pdf["valid_end"].iloc[-1]).date()
assert replay_end_date < pd.Timestamp(exp_config.UNTOUCHED_START_DATE_STR).date()
replay_ohlcv_pdf = complete_ohlcv_pdf[complete_ohlcv_pdf["date"] <= replay_end_date].reset_index(drop=True)
del complete_ohlcv_pdf
# CALL FUNCTION TO CONVERT THE CUT DATA TO ARRAYS, BUILD THE DAILY TABLE AND THE REPLAY PERIODS
ohlcv_array_dict = get_ohlcv_array_dict(replay_ohlcv_pdf)
daily_pdf = get_daily_feature_pdf(ohlcv_array_dict)
period_pdf = get_replay_period_pdf(fold_pdf, ohlcv_array_dict["session_date_list"])
# CHECK THAT THE VALIDATION QUARTERS ARE THOSE OF exp03 (ITS SAVED SCHEDULE, IF PRESENT)
exp03_schedule_path_str = f"{paths.get_experiment_data_path_str('exp03_ath_exit', 'step02_walk_forward_data')}fold_schedule_data.csv"
if os.path.exists(exp03_schedule_path_str):
    exp03_fold_pdf = pd.read_csv(exp03_schedule_path_str)
    same_quarters_bool = exp03_fold_pdf["valid_start"].astype(str).tolist() == fold_pdf["valid_start"].astype(str).tolist() and exp03_fold_pdf["valid_end"].astype(str).tolist() == fold_pdf["valid_end"].astype(str).tolist()
    print(f"Same validation quarters as exp03:\t{same_quarters_bool}")
    assert same_quarters_bool
# SAVE THE SCHEDULE AND THE PERIODS
write_csv_file_to_path(fold_pdf, f"{output_path_str}fold_schedule_data.csv", "W", alert_in=False)
write_csv_file_to_path(period_pdf, f"{output_path_str}replay_period_data.csv", "W", alert_in=False)
# DISPLAY INFORMATION
print(f"Bars given to the simulator:\t{replay_ohlcv_pdf['date'].min()} -> {replay_ohlcv_pdf['date'].max()}\t({len(daily_pdf):,} sessions)")
print(f"Folds:\t{len(fold_pdf)}\tPeriods:\t{len(period_pdf)}\t{period_pdf['period_start'].iloc[0]} -> {period_pdf['period_end'].iloc[-1]}\t({int(period_pdf['session_count'].sum()):,} sessions)")
# PREVIEW DATAFRAME
period_pdf.head()

Same validation quarters as exp03:	True
Bars given to the simulator:	2005-01-03 -> 2026-04-15	(5,353 sessions)
Folds:	44	Periods:	44	2015-04-17 -> 2026-04-15	(2,765 sessions)


,period_id,period_start,period_end,session_count,valid_start,valid_end
0,0,2015-04-17,2015-07-16,63,2015-04-17,2015-07-16
1,1,2015-07-17,2015-10-14,63,2015-07-17,2015-10-16
2,2,2015-10-15,2016-01-15,64,2015-10-15,2016-01-14
3,3,2016-01-19,2016-04-15,62,2016-01-19,2016-04-15
4,4,2016-04-18,2016-07-14,62,2016-04-18,2016-07-15


In [6]:
"""
Candidate paths (AFTER THE FACT: the best of 3 chosen with hindsight is optimistic).
"""
# DEFINE THE CANDIDATES AND THE RULE BUILDER
candidate_list = get_rule_candidate_list()
rule_builder_func = get_rule_builder_func(daily_pdf)
# CALL FUNCTION TO REPLAY EVERY CANDIDATE AND SCORE IT PER PERIOD
start_time = time.time()
candidate_replay_dict = run_candidate_replay_dict(ohlcv_array_dict, period_pdf, candidate_list, rule_builder_func, exp_config.UNTOUCHED_START_DATE_STR,
                                                  exp_config.MAX_CASH_SESSIONS, exp_config.BOOTSTRAP_BLOCK_MONTH_COUNT)
candidate_period_pdf, candidate_summary_pdf = candidate_replay_dict["candidate_period_pdf"], candidate_replay_dict["candidate_summary_pdf"]
print(f"Candidates replayed in {time.time() - start_time:.0f} s")
# LOG EVERY (CANDIDATE, PERIOD) AS A VALIDATION TRIAL (ONE WRITE)
validation_trial_pdf = log_trial_pdf(exp_config, "validation", candidate_period_pdf, KEY_COL_STR_LIST, [col for col in candidate_period_pdf.columns if col.startswith("valid_")],
                                     False, fold_pdf, note_str_in="exp07 step02 candidate path, one period of the continuous replay")
print(f"Validation trials logged:\t{len(validation_trial_pdf)}")
# SAVE THE CANDIDATES
write_csv_file_to_path(candidate_period_pdf, f"{output_path_str}candidate_period_data.csv", "W", alert_in=False)
write_csv_file_to_path(candidate_summary_pdf, f"{output_path_str}candidate_summary_data.csv", "W", alert_in=False)
# PREVIEW DATAFRAME
candidate_summary_pdf[["candidate_idx", "light_k", "strategy_total_return", "buy_hold_total_return", "excess_total_return", "strategy_annualized_return",
                       "buy_hold_annualized_return", "in_market_pct", "exit_count", "mean_cash_sessions", "bought_back_lower_pct", "period_windows_beating_buy_hold",
                       "log_excess_annualized_log_excess", "log_excess_ci_low", "log_excess_ci_high", "strategy_max_drawdown", "buy_hold_max_drawdown"]].round(4)

	candidate 0 {'light_k': 3}: total 95.07% vs buy & hold 235.39% | in market 78.0% | exits 18


	candidate 1 {'light_k': 4}: total 115.60% vs buy & hold 235.39% | in market 84.1% | exits 12


	candidate 2 {'light_k': 5}: total 101.90% vs buy & hold 235.39% | in market 92.4% | exits 8
Candidates replayed in 4 s
Validation trials logged:	132


,candidate_idx,light_k,strategy_total_return,buy_hold_total_return,excess_total_return,strategy_annualized_return,buy_hold_annualized_return,in_market_pct,exit_count,mean_cash_sessions,bought_back_lower_pct,period_windows_beating_buy_hold,log_excess_annualized_log_excess,log_excess_ci_low,log_excess_ci_high,strategy_max_drawdown,buy_hold_max_drawdown
0,0,3,0.9507,2.3539,-1.4032,0.0628,0.1166,0.7804,18,35.1765,0.0556,19,-0.0489,-0.0848,-0.0160,-0.1889,-0.342
1,1,4,1.1560,2.3539,-1.1979,0.0725,0.1166,0.8412,12,36.5833,0.0833,22,-0.0399,-0.0765,-0.0083,-0.2404,-0.342
2,2,5,1.0190,2.3539,-1.3349,0.0661,0.1166,0.9240,8,26.2500,0.0000,13,-0.0458,-0.0953,-0.0127,-0.3153,-0.342


In [7]:
"""
Prior-only path (the honest validation estimate) with its baselines on the same sessions.
"""
# CALL FUNCTION TO SELECT, REPLAY THE SWITCHING PATH AND RUN THE BASELINES
start_time = time.time()
prior_dict = run_prior_only_replay_dict(ohlcv_array_dict, period_pdf, candidate_period_pdf, candidate_list, rule_builder_func, KEY_COL_STR_LIST, TIE_BREAK_LIST,
                                        exp_config.SELECTION_POOLED_QUARTER_COUNT, exp_config.UNTOUCHED_START_DATE_STR, exp_config.MAX_CASH_SESSIONS,
                                        exp_config.RANDOM_RUN_COUNT, exp_config.RANDOM_EXIT_SEED_OFFSET, exp_config.BOOTSTRAP_BLOCK_MONTH_COUNT)
summary_dict, baseline_summary_dict, log_excess_1m_dict = prior_dict["summary_dict"], prior_dict["baseline_summary_dict"], prior_dict["log_excess_1m_dict"]
print(f"Prior-only path and baselines in {time.time() - start_time:.0f} s")
# SAVE THE PATH
write_csv_file_to_path(prior_dict["selection_pdf"], f"{output_path_str}selection_data.csv", "W", alert_in=False)
write_csv_file_to_path(prior_dict["baseline_pdf"], f"{output_path_str}prior_only_baseline_data.csv", "W", alert_in=False)
write_csv_file_to_path(prior_dict["scorecard_pdf"], f"{output_path_str}prior_only_scorecard_data.csv", "W", alert_in=False)
write_csv_file_to_path(prior_dict["simulation_dict"]["daily_equity_pdf"], f"{output_path_str}prior_only_daily_equity_data.csv", "W", alert_in=False)
write_csv_file_to_path(prior_dict["simulation_dict"]["transaction_pdf"], f"{output_path_str}prior_only_transaction_data.csv", "W", alert_in=False)
write_csv_file_to_path(prior_dict["cost_sensitivity_pdf"], f"{output_path_str}prior_only_cost_sensitivity_data.csv", "W", alert_in=False)
write_csv_file_to_path(pd.DataFrame([{key: value for key, value in summary_dict.items() if key != "baselines_beaten"}]), f"{output_path_str}prior_only_summary_data.csv", "W", alert_in=False)
# DISPLAY THE PATH AGAINST BUY AND HOLD
print(f"\nPrior-only path:\t{summary_dict['first_date']} -> {summary_dict['last_date']}\t({summary_dict['session_count']:,} sessions, {summary_dict['year_count']:.2f} years)")
print(f"Total return:\tpath {summary_dict['strategy_total_return']:.2%} | buy & hold {summary_dict['buy_hold_total_return']:.2%}\t(annualized {summary_dict['strategy_annualized_return']:.2%} vs {summary_dict['buy_hold_annualized_return']:.2%})")
print(f"Time in the market:\t{summary_dict['in_market_pct']:.1%}\tExits:\t{prior_dict['simulation_dict']['metric_dict']['signal_exit_count']}\tMean cash decisions per episode:\t{prior_dict['simulation_dict']['metric_dict']['mean_cash_sessions']:.1f}")
print(f"Periods won:\t{summary_dict['period_windows_beating_buy_hold']} of {summary_dict['period_window_count']}\t(sign test p = {summary_dict['period_sign_test_p_value']:.3f})"
      f"\tmean period excess {summary_dict['period_mean_excess_return']:.2%} (SD {summary_dict['period_std_excess_return']:.2%}, t = {summary_dict['period_t_stat']:.2f})"
      f"\tMDE {summary_dict['period_mde_per_window']:.2%} per period ({summary_dict['period_mde_per_year']:.2%} per year)")
print(f"Annualized log excess:\t{summary_dict['log_excess_annualized_log_excess']:.4f}\t95% interval (6-month blocks) [{summary_dict['log_excess_ci_low']:.4f}, {summary_dict['log_excess_ci_high']:.4f}]"
      f"\t(1-month blocks [{log_excess_1m_dict['ci_low']:.4f}, {log_excess_1m_dict['ci_high']:.4f}])\tsupported: {summary_dict['log_excess_supported']}")
print(f"Sharpe:\tpath {summary_dict['strategy_sharpe_ratio']:.2f} | buy & hold {summary_dict['buy_hold_sharpe_ratio']:.2f}\tMax drawdown:\tpath {summary_dict['strategy_max_drawdown']:.2%} | buy & hold {summary_dict['buy_hold_max_drawdown']:.2%}")
print(f"PRIMARY (beats buy & hold):\t{summary_dict['primary_success']}\tComparable:\t{summary_dict['comparable_return']}\tSecondary Sharpe / drawdown:\t{summary_dict['secondary_sharpe']} / {summary_dict['secondary_drawdown']}")
print(f"Random exit (median of {exp_config.RANDOM_RUN_COUNT}):\t{baseline_summary_dict['random_exit_median']:.2%}\tpath beats {baseline_summary_dict['random_exit_path_beats_share']:.0%} of runs")
print(f"Random re-entry (median of {exp_config.RANDOM_RUN_COUNT}):\t{baseline_summary_dict['random_reentry_median']:.2%}\tpath beats {baseline_summary_dict['random_reentry_path_beats_share']:.0%} of runs")
print(f"Matched probabilities:\t{prior_dict['probability_dict']}")
# APPLY THE STOPPING RULE (PROTOCOL.md §9)
information_bool = baseline_summary_dict["random_exit_path_beats_share"] > 0.5 and baseline_summary_dict["random_reentry_path_beats_share"] > 0.5 and summary_dict["all_baselines_beaten"]
verdict_str = "CANDIDATE RESULT PENDING REVIEW" if summary_dict["primary_success"] and information_bool else "STOP"
print(f"\nInformation test passed:\t{information_bool}\tVERDICT:\t{verdict_str}")
# PREVIEW THE COST SENSITIVITY
prior_dict["cost_sensitivity_pdf"].round(4)

Prior-only path and baselines in 43 s

Prior-only path:	2015-07-17 -> 2026-04-15	(2,702 sessions, 10.72 years)
Total return:	path 49.09% | buy & hold 229.74%	(annualized 3.79% vs 11.77%)
Time in the market:	84.4%	Exits:	12	Mean cash decisions per episode:	37.5
Periods won:	18 of 43	(sign test p = 0.889)	mean period excess -1.85% (SD 3.73%, t = -3.25)	MDE 1.59% per period (6.37% per year)
Annualized log excess:	-0.0733	95% interval (6-month blocks) [-0.1255, -0.0349]	(1-month blocks [-0.1228, -0.0246])	supported: False
Sharpe:	path 0.33 | buy & hold 0.71	Max drawdown:	path -31.44% | buy & hold -34.21%
PRIMARY (beats buy & hold):	False	Comparable:	False	Secondary Sharpe / drawdown:	False / False
Random exit (median of 20):	183.31%	path beats 0% of runs
Random re-entry (median of 20):	52.66%	path beats 35% of runs
Matched probabilities:	{'exit_probability': 0.005261210019868324, 'reentry_probability': 0.02850356294536817}

Information test passed:	False	VERDICT:	STOP


,slippage_per_share,total_return,buy_hold_return,excess_return
0,0.00,0.4917,2.2975,-1.8058
1,0.01,0.4909,2.2974,-1.8065
2,0.02,0.4892,2.2973,-1.8082


In [8]:
"""
Episode scorecard of the prior-only path (every exit). S / R > 1 = bought back lower than sold.
"""
# DISPLAY THE SCORECARD
scorecard_pdf = prior_dict["scorecard_pdf"]
if not scorecard_pdf.empty:
    print(f"Episodes:\t{len(scorecard_pdf)}\tbought back lower:\t{int((scorecard_pdf['s_over_r'] > 1).sum())}\tproduct of S/R:\t{np.prod(scorecard_pdf['s_over_r']):.4f}"
          f"\tmean sessions out:\t{scorecard_pdf['sessions_out'].mean():.1f}\ttotal cost:\t{scorecard_pdf['cost_paid_pct'].sum():.4%} of the sale values")
scorecard_pdf[["exit_date", "exit_fill_price", "reentry_date", "reentry_fill_price", "s_over_r", "sessions_out", "buy_hold_return_out", "max_decline_out_pct", "cost_paid_pct", "open_at_end"]].round(4)

Episodes:	12	bought back lower:	0	product of S/R:	0.4521	mean sessions out:	35.1	total cost:	0.1157% of the sale values


,exit_date,exit_fill_price,reentry_date,reentry_fill_price,s_over_r,sessions_out,buy_hold_return_out,max_decline_out_pct,cost_paid_pct,open_at_end
0,2015-08-24,190.17,2015-09-25,192.99,0.9854,23,0.0147,-0.0164,0.0002,False
1,2016-01-08,191.73,2016-03-11,202.81,0.9454,43,0.0577,-0.0559,0.0002,False
2,2018-10-29,263.36,2018-11-07,280.96,0.9374,7,0.0668,-0.0009,0.0001,False
3,2018-11-20,264.28,2018-11-26,267.43,0.9882,3,0.0118,-0.0046,0.0001,False
4,2018-12-07,263.76,2019-02-12,274.08,0.9623,44,0.0390,-0.1138,0.0001,False
5,2019-05-31,275.47,2019-06-12,288.19,0.9559,8,0.0461,-0.0087,0.0001,False
6,2020-03-19,240.99,2020-05-27,303.50,0.7940,47,0.2593,-0.0944,0.0001,False
7,2022-03-10,425.93,2022-03-22,449.39,0.9478,8,0.0550,-0.0238,0.0001,False
8,2022-05-10,399.38,2022-08-08,413.03,0.9670,61,0.0341,-0.0932,0.0001,False
9,2022-10-05,376.69,2023-05-01,416.00,0.9055,142,0.1043,-0.0759,0.0001,False


In [9]:
# DISPLAY THE SELECTED CANDIDATE PER PERIOD (THE CHOICE OF PERIOD f GOVERNS PERIOD f + 1)
selection_pdf = prior_dict["selection_pdf"]
print("Selections (candidate: periods):", selection_pdf.groupby(["light_k"]).size().to_dict())
selection_pdf[["fold_id", "light_k", "pooled_score", "pooled_quarter_count"]].tail(12).round(4)

Selections (candidate: periods): {3: 12, 4: 17, 5: 15}


,fold_id,light_k,pooled_score,pooled_quarter_count
32,32,3,-0.0254,4
33,33,5,-0.0273,4
34,34,5,-0.0001,4
35,35,5,-0.0001,4
36,36,4,-0.0000,4
37,37,4,-0.0000,4
38,38,4,-0.0000,4
39,39,3,0.0170,4
40,40,3,-0.0119,4
41,41,3,-0.0119,4


In [10]:
# CREATE A FIGURE OF THE PRIOR-ONLY PATH vs BUY AND HOLD (GROWTH OF 1)
path_equity_pdf, buy_hold_equity_pdf = prior_dict["simulation_dict"]["daily_equity_pdf"], prior_dict["buy_hold_dict"]["daily_equity_pdf"]
fig = go.Figure()
fig.add_trace(go.Scatter(x=path_equity_pdf["date"], y=path_equity_pdf["equity"] / config.INITIAL_CAPITAL, name="prior-only path", line={"width": 1.5}))
fig.add_trace(go.Scatter(x=buy_hold_equity_pdf["date"], y=buy_hold_equity_pdf["equity"] / config.INITIAL_CAPITAL, name="buy & hold", line={"width": 1.5}))
fig.update_layout(template="plotly_dark", width=1100, height=450, title="exp07 prior-only path vs buy-and-hold (continuous replay, growth of 1)")
fig.show()

In [11]:
# LOG ONE SUMMARY ENTRY OF THE RUN (THE CANDIDATE PERIODS ARE ALREADY LOGGED; THIS RECORDS THE HONEST ESTIMATE THAT WAS SEEN)
summary_trial_dict = log_trial_dict(exp_config, "summary", {"run_mode": RUN_MODE_STR, "selection": f"pooled {exp_config.SELECTION_POOLED_QUARTER_COUNT} periods", "path": "prior-only continuous replay"},
                                    {**{key: value for key, value in summary_dict.items() if key != "baselines_beaten"}, **baseline_summary_dict,
                                     "log_excess_1m_ci_low": log_excess_1m_dict["ci_low"], "log_excess_1m_ci_high": log_excess_1m_dict["ci_high"],
                                     "exit_count": prior_dict["simulation_dict"]["metric_dict"]["signal_exit_count"], "verdict": verdict_str,
                                     "best_after_the_fact_total_return": float(candidate_summary_pdf["strategy_total_return"].max())},
                                    False, note_str_in="exp07 step02 validation summary (prior-only continuous path, baselines, MDE, log-excess interval)")
print(f"Summary logged:\t{summary_trial_dict['trial_id']}")

Summary logged:	20261006145605946803_2965fad3e3


In [12]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)

,experiment,stage,trial_count,config_count,test_touched_count,first_logged_ts,last_logged_ts
0,exp01_minute_entry,signal_check,2,2,0,2026-10-05T15:17:38,2026-10-06T12:46:46
1,exp01_minute_entry,summary,1,1,0,2026-10-05T16:16:30,2026-10-05T16:16:30
2,exp01_minute_entry,validation,540,1,0,2026-10-05T15:36:53,2026-10-05T16:16:18
3,exp02_stop_reentry,exploration,14,1,0,2026-10-05T14:47:50,2026-10-05T14:47:50
4,exp02_stop_reentry,signal_check,1,1,0,2026-10-05T14:48:41,2026-10-05T14:48:41
5,exp02_stop_reentry,summary,1,1,0,2026-10-05T14:50:29,2026-10-05T14:50:29
6,exp02_stop_reentry,validation,792,1,0,2026-10-05T14:49:18,2026-10-05T14:49:48
7,exp03_ath_exit,exploration,15,1,0,2026-10-05T14:55:04,2026-10-05T14:55:04
8,exp03_ath_exit,summary,1,1,0,2026-10-05T14:56:49,2026-10-05T14:56:49
9,exp03_ath_exit,validation,660,1,0,2026-10-05T14:55:55,2026-10-05T14:56:12
